# 📚 GUÍA COMPLETA DE DATAFRAMES EN SPARK

## 🎯 **OBJETIVO**
Guía exhaustiva de todos los métodos principales de DataFrames en PySpark con ejemplos prácticos.

## 📋 **CONTENIDO**
- 🔍 **Selección y Filtrado**: select, filter, where, drop, distinct
- 🔗 **Joins**: inner, left, right, full, cross joins
- 📊 **Agregaciones**: groupBy, agg, pivot, rollup, cube
- 🔄 **Transformaciones**: withColumn, dropDuplicates, union, intersect
- 🎯 **Ordenamiento**: orderBy, sort, repartition, coalesce
- 💾 **Persistencia**: cache, persist, checkpoint
- 📤 **E/S**: read, write, show, collect, take

---

## 🔧 **CONFIGURACIÓN INICIAL**


In [ ]:
# 🔄 CELDA DE REINICIO - Ejecutar si hay errores de SparkContext
try:
    if 'spark' in globals():
        print("🔄 Cerrando sesión anterior de Spark...")
        spark.stop()
        print("✅ Sesión anterior cerrada")
except:
    print("ℹ️ No había sesión anterior")

if 'spark' in globals():
    del spark

print("🚀 Listo para crear nueva sesión de Spark")


In [ ]:
# Importar todas las librerías necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import socket
import os

print("📚 Librerías importadas correctamente")


In [ ]:
# Crear SparkSession optimizada
def get_spark_master():
    try:
        hostname = socket.gethostname()
        if 'jupyter' in hostname or 'master' in hostname or 'jupyterlab' in hostname:
            return "spark://master:7077"
        else:
            return "spark://localhost:7077"
    except:
        return "local[*]"

spark_master_url = get_spark_master()
print(f"🔧 Conectando a: {spark_master_url}")

spark = SparkSession.builder \
    .appName("Guia-DataFrames-Complete") \
    .master(spark_master_url) \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.instances", "1") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

print("✅ SparkSession creada exitosamente")


In [ ]:
# Crear datos de ejemplo para todas las demostraciones
print("🏗️ Creando datasets de ejemplo...")

# Dataset 1: Empleados
empleados_data = [
    (1, "Juan Pérez", "IT", 50000, "Madrid"),
    (2, "María García", "Marketing", 45000, "Barcelona"),
    (3, "Carlos López", "IT", 55000, "Madrid"),
    (4, "Ana Martín", "HR", 40000, "Valencia"),
    (5, "Luis Rodríguez", "IT", 60000, "Sevilla"),
    (6, "Laura Sánchez", "Marketing", 48000, "Barcelona"),
    (7, "Pedro González", "Sales", 42000, "Madrid"),
    (8, "Carmen Díaz", "IT", 52000, "Valencia")
]

empleados_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("departamento", StringType(), True),
    StructField("salario", IntegerType(), True),
    StructField("ciudad", StringType(), True)
])

df_empleados = spark.createDataFrame(empleados_data, empleados_schema)

# Dataset 2: Departamentos
departamentos_data = [
    ("IT", "Tecnología", "Edificio A"),
    ("Marketing", "Comercial", "Edificio B"),
    ("HR", "Recursos Humanos", "Edificio C"),
    ("Sales", "Ventas", "Edificio A")
]

df_departamentos = spark.createDataFrame(departamentos_data, 
                                       ["codigo", "nombre_completo", "ubicacion"])

# Dataset 3: Proyectos
proyectos_data = [
    (1, "Sistema Web", "IT", 100000),
    (2, "Campaña Digital", "Marketing", 50000),
    (3, "Reclutamiento", "HR", 30000),
    (4, "CRM Nuevo", "Sales", 80000),
    (5, "App Móvil", "IT", 120000)
]

df_proyectos = spark.createDataFrame(proyectos_data,
                                   ["proyecto_id", "nombre_proyecto", "departamento", "presupuesto"])

print("✅ Datasets creados exitosamente")
print(f"📊 Empleados: {df_empleados.count()} registros")
print(f"🏢 Departamentos: {df_departamentos.count()} registros") 
print(f"📋 Proyectos: {df_proyectos.count()} registros")


## 🔍 **SECCIÓN 1: SELECCIÓN Y FILTRADO**

### **📋 Métodos principales:**
- `select()` - Seleccionar columnas
- `filter()` / `where()` - Filtrar filas
- `drop()` - Eliminar columnas  
- `distinct()` - Eliminar duplicados
- `sample()` - Muestreo aleatorio


In [ ]:
# 🔍 SELECT - Seleccionar columnas
print("=" * 60)
print("🔍 MÉTODO: select() - Seleccionar columnas específicas")
print("=" * 60)

# Seleccionar columnas específicas
df_select1 = df_empleados.select("nombre", "departamento", "salario")
print("📋 Seleccionar columnas por nombre:")
df_select1.show()

# Seleccionar con alias
df_select2 = df_empleados.select(
    col("nombre").alias("empleado"),
    col("salario").alias("sueldo_anual")
)
print("📋 Seleccionar con alias:")
df_select2.show()

# Seleccionar todas las columnas excepto una
df_select3 = df_empleados.select("*").drop("ciudad")
print("📋 Todas las columnas excepto 'ciudad':")
df_select3.show()

# Seleccionar con expresiones
df_select4 = df_empleados.select(
    col("nombre"),
    col("salario"),
    (col("salario") * 1.1).alias("salario_con_bonus")
)
print("📋 Seleccionar con expresiones calculadas:")
df_select4.show()


In [ ]:
# 🔍 FILTER/WHERE - Filtrar filas
print("=" * 60)
print("🔍 MÉTODO: filter() y where() - Filtrar filas")
print("=" * 60)

# Filter con condiciones simples
df_filter1 = df_empleados.filter(col("salario") > 50000)
print("📋 Empleados con salario > 50000:")
df_filter1.show()

# Filter con múltiples condiciones
df_filter2 = df_empleados.filter(
    (col("departamento") == "IT") & (col("salario") >= 50000)
)
print("📋 Empleados de IT con salario >= 50000:")
df_filter2.show()

# Where (sinónimo de filter)
df_where1 = df_empleados.where(col("ciudad").isin(["Madrid", "Barcelona"]))
print("📋 Empleados en Madrid o Barcelona (usando where):")
df_where1.show()

# Filter con isNull/isNotNull
df_empleados_con_nulos = df_empleados.union(
    spark.createDataFrame([(9, "Test Null", None, 30000, "Test")], empleados_schema)
)
df_filter_nulls = df_empleados_con_nulos.filter(col("departamento").isNotNull())
print("📋 Empleados con departamento no nulo:")
df_filter_nulls.show()

# Filter con like/regex
df_filter_like = df_empleados.filter(col("nombre").like("%García%"))
print("📋 Empleados con 'García' en el nombre:")
df_filter_like.show()


## 🔗 **SECCIÓN 2: JOINS**

### **📋 Tipos de joins:**
- `join()` - Inner join por defecto
- `inner` - Solo registros que coinciden en ambas tablas
- `left` - Todos los registros de la tabla izquierda
- `right` - Todos los registros de la tabla derecha
- `full` / `outer` - Todos los registros de ambas tablas
- `cross` - Producto cartesiano


In [ ]:
# 🔗 JOINS - Unir DataFrames
print("=" * 60)
print("🔗 MÉTODO: join() - Unir DataFrames")
print("=" * 60)

# Inner Join (por defecto)
df_inner = df_empleados.join(
    df_departamentos, 
    df_empleados.departamento == df_departamentos.codigo,
    "inner"
).select("nombre", "departamento", "salario", "nombre_completo", "ubicacion")

print("📋 INNER JOIN - Empleados con información de departamento:")
df_inner.show()

# Left Join
df_left = df_empleados.join(
    df_proyectos,
    df_empleados.departamento == df_proyectos.departamento,
    "left"
).select("nombre", "departamento", "salario", "nombre_proyecto", "presupuesto")

print("📋 LEFT JOIN - Empleados con proyectos (puede tener nulls):")
df_left.show()

# Right Join
df_right = df_empleados.join(
    df_proyectos,
    df_empleados.departamento == df_proyectos.departamento,
    "right"
).select("nombre", "departamento", "salario", "nombre_proyecto", "presupuesto")

print("📋 RIGHT JOIN - Proyectos con empleados (puede tener nulls):")
df_right.show()

# Full Outer Join
df_full = df_empleados.join(
    df_proyectos,
    df_empleados.departamento == df_proyectos.departamento,
    "full"
).select("nombre", "departamento", "salario", "nombre_proyecto", "presupuesto")

print("📋 FULL OUTER JOIN - Todos los registros de ambas tablas:")
df_full.show()


## 📊 **SECCIÓN 3: AGREGACIONES**

### **📋 Métodos principales:**
- `groupBy()` - Agrupar por columnas
- `agg()` - Funciones de agregación
- `pivot()` - Pivotar datos
- `rollup()` - Agregaciones jerárquicas
- `cube()` - Agregaciones multidimensionales


In [ ]:
# 📊 GROUPBY y AGG - Agregaciones
print("=" * 60)
print("📊 MÉTODO: groupBy() y agg() - Agregaciones")
print("=" * 60)

# Agregación básica por departamento
df_agg1 = df_empleados.groupBy("departamento").agg(
    count("nombre").alias("total_empleados"),
    avg("salario").alias("salario_promedio"),
    max("salario").alias("salario_maximo"),
    min("salario").alias("salario_minimo"),
    sum("salario").alias("salario_total")
)

print("📋 Estadísticas por departamento:")
df_agg1.show()

# Agregación por múltiples columnas
df_agg2 = df_empleados.groupBy("departamento", "ciudad").agg(
    count("nombre").alias("empleados_por_ciudad"),
    avg("salario").alias("salario_promedio_ciudad")
)

print("📋 Estadísticas por departamento y ciudad:")
df_agg2.show()

# Agregación con filtros
df_agg3 = df_empleados.filter(col("salario") > 45000).groupBy("departamento").agg(
    count("nombre").alias("empleados_alto_salario"),
    avg("salario").alias("salario_promedio_alto")
)

print("📋 Empleados con salario > 45000 por departamento:")
df_agg3.show()


## 🔄 **SECCIÓN 4: TRANSFORMACIONES**

### **📋 Métodos principales:**
- `withColumn()` - Agregar/modificar columnas
- `withColumnRenamed()` - Renombrar columnas
- `drop()` - Eliminar columnas
- `dropDuplicates()` - Eliminar duplicados
- `union()` - Combinar DataFrames
- `intersect()` - Intersección de DataFrames


In [ ]:
# 🔄 TRANSFORMACIONES - Modificar DataFrames
print("=" * 60)
print("🔄 MÉTODO: withColumn() - Agregar/modificar columnas")
print("=" * 60)

# Agregar columnas calculadas
df_transform1 = df_empleados.withColumn("salario_mensual", col("salario") / 12) \
                           .withColumn("categoria_salario", 
                                      when(col("salario") >= 55000, "Alto")
                                      .when(col("salario") >= 45000, "Medio")
                                      .otherwise("Bajo")) \
                           .withColumn("inicial_nombre", split(col("nombre"), " ")[0])

print("📋 DataFrame con columnas calculadas:")
df_transform1.select("nombre", "salario", "salario_mensual", "categoria_salario", "inicial_nombre").show()

# Renombrar columnas
df_transform2 = df_empleados.withColumnRenamed("nombre", "empleado") \
                           .withColumnRenamed("departamento", "depto")

print("📋 DataFrame con columnas renombradas:")
df_transform2.show()

# Eliminar duplicados
df_con_duplicados = df_empleados.union(
    spark.createDataFrame([(1, "Juan Pérez", "IT", 50000, "Madrid")], empleados_schema)
)
df_sin_duplicados = df_con_duplicados.dropDuplicates()

print("📋 DataFrame original con duplicados:")
df_con_duplicados.show()
print("📋 DataFrame sin duplicados:")
df_sin_duplicados.show()

# Union de DataFrames
nuevos_empleados = spark.createDataFrame([
    (9, "Roberto Silva", "IT", 48000, "Madrid"),
    (10, "Patricia Ruiz", "Marketing", 46000, "Barcelona")
], empleados_schema)

df_union = df_empleados.union(nuevos_empleados)
print("📋 DataFrame con nuevos empleados (union):")
df_union.show()


## 🎯 **SECCIÓN 5: ORDENAMIENTO Y PERSISTENCIA**

### **📋 Métodos principales:**
- `orderBy()` / `sort()` - Ordenar datos
- `repartition()` - Redistribuir particiones
- `coalesce()` - Reducir particiones
- `cache()` / `persist()` - Almacenar en memoria
- `unpersist()` - Liberar memoria


In [ ]:
# 🎯 ORDENAMIENTO Y PERSISTENCIA
print("=" * 60)
print("🎯 MÉTODO: orderBy() y sort() - Ordenar datos")
print("=" * 60)

# Ordenamiento simple
df_orden1 = df_empleados.orderBy("salario")
print("📋 Empleados ordenados por salario (ascendente):")
df_orden1.show()

# Ordenamiento descendente
df_orden2 = df_empleados.orderBy(col("salario").desc())
print("📋 Empleados ordenados por salario (descendente):")
df_orden2.show()

# Ordenamiento múltiple
df_orden3 = df_empleados.orderBy("departamento", col("salario").desc())
print("📋 Empleados ordenados por departamento y salario desc:")
df_orden3.show()

# Sort (sinónimo de orderBy)
df_sort = df_empleados.sort("nombre")
print("📋 Empleados ordenados por nombre (usando sort):")
df_sort.show()

print("=" * 60)
print("💾 MÉTODO: cache() y persist() - Almacenar en memoria")
print("=" * 60)

# Cache un DataFrame que usaremos múltiples veces
df_empleados_cached = df_empleados.cache()
print("✅ DataFrame cachead en memoria")

# Verificar que está cachead
print(f"📊 Número de particiones: {df_empleados_cached.rdd.getNumPartitions()}")
print(f"📊 Número de registros: {df_empleados_cached.count()}")

# Liberar memoria
df_empleados_cached.unpersist()
print("🗑️ Memoria liberada (unpersist)")


## 📤 **SECCIÓN 6: MÉTODOS DE SALIDA**

### **📋 Métodos principales:**
- `show()` - Mostrar datos en consola
- `collect()` - Obtener todos los datos como lista
- `take()` - Obtener N primeros registros
- `head()` - Obtener primeros registros
- `first()` - Obtener primer registro
- `count()` - Contar registros
- `describe()` - Estadísticas descriptivas


In [ ]:
# 📤 MÉTODOS DE SALIDA
print("=" * 60)
print("📤 MÉTODOS DE SALIDA - Obtener y mostrar datos")
print("=" * 60)

# Show con diferentes parámetros
print("📋 show() - Mostrar datos:")
df_empleados.show()

print("📋 show(5) - Mostrar solo 5 registros:")
df_empleados.show(5)

print("📋 show(3, False) - Mostrar 3 registros sin truncar:")
df_empleados.show(3, False)

# Take - Obtener N registros como lista
print("📋 take(3) - Obtener 3 registros como lista:")
registros = df_empleados.take(3)
for i, registro in enumerate(registros):
    print(f"  {i+1}: {registro}")

# Head - Primeros registros
print("📋 head(2) - Primeros 2 registros:")
primeros = df_empleados.head(2)
for registro in primeros:
    print(f"  {registro}")

# First - Primer registro
print("📋 first() - Primer registro:")
primer = df_empleados.first()
print(f"  {primer}")

# Count - Contar registros
print(f"📊 count() - Total de registros: {df_empleados.count()}")

# Describe - Estadísticas descriptivas
print("📊 describe() - Estadísticas descriptivas:")
df_empleados.describe().show()

# PrintSchema - Mostrar esquema
print("📋 printSchema() - Estructura del DataFrame:")
df_empleados.printSchema()


## 🎯 **RESUMEN DE MÉTODOS DE DATAFRAMES**

### **📚 Métodos aprendidos en esta guía:**

#### **🔍 Selección y Filtrado:**
- `select()` - Seleccionar columnas específicas
- `filter()` / `where()` - Filtrar filas con condiciones
- `drop()` - Eliminar columnas
- `distinct()` - Eliminar duplicados

#### **🔗 Joins:**
- `join()` - Unir DataFrames (inner, left, right, full, cross)

#### **📊 Agregaciones:**
- `groupBy()` - Agrupar datos
- `agg()` - Funciones de agregación (count, sum, avg, max, min)

#### **🔄 Transformaciones:**
- `withColumn()` - Agregar/modificar columnas
- `withColumnRenamed()` - Renombrar columnas
- `dropDuplicates()` - Eliminar duplicados
- `union()` - Combinar DataFrames

#### **🎯 Ordenamiento y Persistencia:**
- `orderBy()` / `sort()` - Ordenar datos
- `cache()` / `persist()` - Almacenar en memoria
- `unpersist()` - Liberar memoria

#### **📤 Salida:**
- `show()` - Mostrar datos
- `collect()` - Obtener todos los datos
- `take()` - Obtener N registros
- `count()` - Contar registros
- `describe()` - Estadísticas descriptivas

---

## 💡 **CONSEJOS PARA EL ÉXITO**

### **🎯 Mejores Prácticas:**
1. **Usa `select()`** para limitar columnas y mejorar rendimiento
2. **Filtra temprano** con `filter()` antes de agregaciones
3. **Cache DataFrames** que usarás múltiples veces
4. **Usa joins apropiados** según tus necesidades
5. **Ordena solo cuando sea necesario** (puede ser costoso)

### **🚀 Próximos pasos:**
1. **Practica** con tus propios datos
2. **Experimenta** con diferentes combinaciones
3. **Optimiza** consultas complejas
4. **Explora** Window Functions en el tutorial avanzado

---

**🎉 ¡Has dominado los métodos principales de DataFrames en Spark!**


In [ ]:
# 🔒 Cerrar SparkSession
spark.stop()
print("🔒 SparkSession cerrada correctamente")
print("🎉 ¡Guía completa de DataFrames finalizada!")
